# systemgmmkit quickstart

This Kaggle/Colab notebook is package-scoped: it demonstrates `systemgmmkit` panel-data and post-estimation utilities only. It is intended as a lightweight reproducibility and adoption artifact, not a cross-package benchmark.

## Install

Kaggle and Colab sessions start from a clean runtime, so install the published package first.

In [ ]:
%pip install -q systemgmmkit

## Fit a small panel-data workflow

The example builds a deterministic panel, fits a robust OLS specification, and runs post-estimation diagnostics.

In [ ]:
import numpy as np
import pandas as pd

import systemgmmkit as sgk

rows = []
for firm in range(1, 9):
    for year in range(1, 10):
        investment = 0.2 * year + 0.1 * firm
        leverage = 0.5 + 0.03 * firm
        growth = 1.0 + 2.0 * investment - 0.5 * leverage
        rows.append({
            "firm": firm,
            "year": year,
            "growth": growth,
            "investment": investment,
            "leverage": leverage,
        })

df = pd.DataFrame(rows)

spec = sgk.OLSSpec(
    dependent="growth",
    regressors=["investment", "leverage"],
    covariance="robust",
)
result = sgk.run_ols(spec, df)

post = sgk.quick_postestimation(
    result,
    df,
    y="growth",
    lincoms={"total_effect": "investment + leverage"},
    wald_tests={"joint_zero": "investment = 0, leverage = 0"},
)

display(result.params.round(4).to_frame("estimate"))
display(pd.Series(post.metrics).round(6).to_frame("value"))
display(post.linear_combinations.round(4))
display(post.wald_tests.round(4))

assert np.isfinite(post.metrics["rmse"])
assert post.linear_combinations is not None
assert post.wald_tests is not None